# R27_v2 — Observability Audit dei v2 (R25_v2 + R26_v2 + R28_v2 + R29_v2)

Audit retro-attivo dei run v2 con metriche estese (T_intra_corr + rank_effective + cond_number). Stessa logica di R27 originale ma su run con baseline R24F stabile (gn_max < 25).

Output: `results/Prodigy_Study/R27_Audit_v2/audit_summary.csv` + JSON per run.

In [ ]:
# Cell 1 -- Bootstrap + GLOBALS
import sys, os, subprocess
RESULTS_DIR = 'results/Prodigy_Study/R27_Audit_v2'
OUTPUT_CSV = f'{RESULTS_DIR}/audit_summary.csv'
BRANCH = 'Prodigy_Deep_Study'
os.makedirs(RESULTS_DIR, exist_ok=True)
sys.path.insert(0, '.')
from scripts.audit_checkpoints import discover_runs, find_checkpoint
br = subprocess.run(['git','branch','--show-current'], capture_output=True, text=True).stdout.strip()
assert br == BRANCH
print(f'  [OK] branch={br}')

In [ ]:
# Cell 2 -- Audit (idempotente: skip se CSV con righe gia' esistente)
import os, pandas as pd, subprocess, sys
FORCE_RERUN = False
if os.path.isfile(OUTPUT_CSV) and not FORCE_RERUN:
    try:
        _df = pd.read_csv(OUTPUT_CSV)
        if len(_df) > 0:
            print(f'[SKIP] CSV gia\' esistente con {len(_df)} righe. FORCE_RERUN=True per riprocessare.')
            skip = True
        else: skip = False
    except: skip = False
else:
    skip = False
if not skip:
    # Pattern include R30 (FIX 2026-06-12: prima escludeva R30).
    cmd = [sys.executable, '-u', 'scripts/audit_checkpoints.py',
           '--results_root', 'results/Prodigy_Study',
           '--output_csv', OUTPUT_CSV,
           '--pattern', r'^R(25v2|26v2|28v2|29v2|30)_',
           '--device', 'cpu']
    print(f'\nRunning: {" ".join(cmd)}\n')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding='utf-8', errors='replace', bufsize=1)
    for line in proc.stdout: print(line, end='', flush=True)
    proc.wait()
    assert proc.returncode == 0
df = pd.read_csv(OUTPUT_CSV)
print(f'\n[DONE] {len(df)} run auditati.')

In [ ]:
# Cell 3 -- Summary
import pandas as pd
df = pd.read_csv(OUTPUT_CSV)
print(f'Tot: {len(df)}')
print(f'Best T_intra: {df.sort_values("val_val_T_intra_corr", ascending=False).iloc[0]["tag"]}')
print(f'Best val_data: {df.sort_values("val_data", ascending=True).iloc[0]["tag"]}')
print(df.sort_values('val_val_T_intra_corr', ascending=False)[
    ['tag','val_data','val_val_T_tracking_corr','val_val_T_intra_corr','rank_effective']
].round(4).to_string(index=False))